<a href="https://colab.research.google.com/github/DianaBravoPerez/EDP-1/blob/main/servidor_markov_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cadena de Markov en Tiempo Continuo — Modelo de Servidor

**Estados:**
- 0 → libre
- 1 → ocupado
- 2 → caído

**Fórmula principal:** $P(t) = e^{tQ}$

## Librerías necesarias

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import random
from scipy.linalg import expm

## Definición de la matriz $Q$

> Añadir blockquote



In [ ]:
t = sp.Symbol('t', positive=True)

Q = sp.Matrix([
    [-2,                2,   0                ],
    [ 3,               -4,   1                ],
    [sp.Rational(1,2),  0,  sp.Rational(-1,2) ]
])

sp.pprint(Q)

## Cálculo de $P(t) = e^{tQ}$ con SymPy

*(Puede tardar unos segundos)*

In [ ]:
Pt = (t * Q).exp()
sp.pprint(Pt)

## Pregunta 1
**Si el sistema inicia libre (estado 0), ¿cuál es la probabilidad de estar caído (estado 2) en $t=1$?**

Se evalúa la entrada $[0,\, 2]$ de $P(1)$.

In [ ]:
P1 = Pt.subs(t, 1)

respuesta = P1[0, 2]

print('Probabilidad de estar caído en t=1:')
print(float(respuesta.evalf()))

## Pregunta 2
**¿Cómo cambia $P(t)$ cuando $t \\to \\infty$?**

Converge a la distribución estacionaria $\\pi$, que satisface $\\pi Q = 0$.

In [ ]:
pi0, pi1, pi2 = sp.symbols('pi0 pi1 pi2', nonneg=True)

# Ecuaciones: pi * Q = 0  y  pi0 + pi1 + pi2 = 1
ec1 = -2*pi0 + 3*pi1 + sp.Rational(1,2)*pi2
ec2 =  2*pi0 - 4*pi1
ec3 =       pi1 - sp.Rational(1,2)*pi2
ec4 = pi0 + pi1 + pi2 - 1

sol = sp.solve([ec1, ec2, ec3, ec4], [pi0, pi1, pi2])

print('pi_0 (libre)   =', sol[pi0])
print('pi_1 (ocupado) =', sol[pi1])
print('pi_2 (caído)   =', sol[pi2])

## Pregunta 3
**Gráfica de $P_{00}(t)$, $P_{01}(t)$, $P_{02}(t)$**

In [ ]:
Q_num = np.array([[-2,  2,    0  ],
                   [ 3, -4,    1  ],
                   [0.5, 0,  -0.5]])

tiempos = np.linspace(0, 10, 200)

P00 = []
P01 = []
P02 = []

for ti in tiempos:
    fila = expm(ti * Q_num)[0]   # primera fila de P(ti)
    P00.append(fila[0])
    P01.append(fila[1])
    P02.append(fila[2])

plt.figure(figsize=(8, 4))
plt.plot(tiempos, P00, label='libre  P00')
plt.plot(tiempos, P01, label='ocupado P01')
plt.plot(tiempos, P02, label='caído  P02')
plt.xlabel('t')
plt.ylabel('Probabilidad')
plt.title('P(t) partiendo del estado libre')
plt.legend()
plt.grid(True)
plt.show()

## Pregunta 4
**Simulación de trayectorias**

En cada estado $i$ el sistema espera un tiempo $\\sim \\text{Exp}(-Q_{ii})$
y luego salta al estado $j$ con probabilidad $Q_{ij}/(-Q_{ii})$.

In [ ]:
Q_lista = [[-2, 2, 0],
            [ 3,-4, 1],
            [0.5, 0,-0.5]]

def simular(estado_inicial, T_max):
    estado  = estado_inicial
    tiempo  = 0.0
    hist_t  = [0.0]
    hist_e  = [estado]

    while tiempo < T_max:
        tasa = -Q_lista[estado][estado]
        dt   = random.expovariate(tasa)
        tiempo = tiempo + dt
        if tiempo > T_max:
            break
        # elegir siguiente estado
        if estado == 0:
            siguiente = 1          # único destino desde 0
        elif estado == 1:
            u = random.random()
            siguiente = 0 if u < 3/4 else 2
        else:                      # estado == 2
            siguiente = 0
        estado = siguiente
        hist_t.append(tiempo)
        hist_e.append(estado)

    return hist_t, hist_e

In [ ]:
# Se grafican 6 trayectorias de ejemplo
plt.figure(figsize=(9, 4))

for _ in range(6):
    tt, ee = simular(0, 10)
    plt.step(tt, ee, where='post', alpha=0.7)

plt.yticks([0, 1, 2], ['libre', 'ocupado', 'caído'])
plt.xlabel('Tiempo')
plt.title('Trayectorias simuladas (inicio: libre)')
plt.grid(True)
plt.show()

### Comparación: simulado vs analítico

Se estima $P(X(t)=j \\mid X(0)=0)$ promediando 300 trayectorias.

In [ ]:
N = 300
t_grid = np.linspace(0, 10, 80)

conteo = [0]*len(t_grid)   # cuántas trayectorias están en estado 2 (caído)

for _ in range(N):
    tt, ee = simular(0, 10)
    for k in range(len(t_grid)):
        tk = t_grid[k]
        # estado en el tiempo tk
        idx = 0
        for i in range(len(tt)):
            if tt[i] <= tk:
                idx = i
        if ee[idx] == 2:
            conteo[k] += 1

prob_caido_sim = [c / N for c in conteo]

# Curva analítica P02(t)
P02_analitica = [expm(ti * Q_num)[0, 2] for ti in t_grid]

plt.figure(figsize=(7, 4))
plt.plot(t_grid, P02_analitica, label='Analítico P02(t)', linewidth=2)
plt.plot(t_grid, prob_caido_sim, '--', label='Simulado (N=300)')
plt.xlabel('t')
plt.ylabel('P(caído | inicio libre)')
plt.title('Analítico vs Simulado — estado caído')
plt.legend()
plt.grid(True)
plt.show()